# Download PR review comment dataset

Pipeline: GH Archive → filter by top snapshot commits → GraphQL enrichment (`pr_title`, `pr_body`, `repo_star_count`, `is_resolved`) → unified code enrichment (compare patches, base/snapshot zipballs, patched content, dependency resolution).

Set `GITHUB_TOKEN` in `.env` (or the environment) before enrichment steps. Install the package editable: `pip install -e .` from the repo root.

In [ ]:
import asyncio
import json
import logging

import aiohttp
from dotenv import load_dotenv

from ai_code_reviewer.dataset import (
    checkpoints,
    gh_archive,
    github_api,
    github_graphql,
)
from ai_code_reviewer.dataset import config

load_dotenv()
logging.basicConfig(level=logging.INFO)

In [ ]:
gh_archive_semaphore = asyncio.Semaphore(config.GH_ARCHIVE_CONCURRENCY)
gh_semaphore = asyncio.Semaphore(config.GITHUB_API_CONCURRENCY)

In [ ]:
async with aiohttp.ClientSession(
    connector=config.tcp_connector_for_concurrency(config.GH_ARCHIVE_CONCURRENCY),
    timeout=aiohttp.ClientTimeout(
        total=config.HTTP_JSON_TIMEOUT_TOTAL,
        connect=config.HTTP_JSON_TIMEOUT_CONNECT,
        sock_connect=config.HTTP_JSON_TIMEOUT_SOCK_CONNECT,
        sock_read=config.HTTP_JSON_TIMEOUT_SOCK_READ,
    ),
) as session:
    dataset = await gh_archive.fetch_pr_comments_range(
        session,
        config.RANGE_START,
        config.RANGE_END,
        gh_archive_semaphore,
    )

checkpoints.save_dataset_checkpoint(dataset, config.DATASET_RAW_PATH)

In [ ]:
dataset = gh_archive.filter_dataset_by_top_snapshot_commits(
    dataset,
    4#config.SNAPSHOT_COMMITS_TO_KEEP,
)

checkpoints.save_dataset_checkpoint(dataset, config.DATASET_FILTERED_PATH)

In [ ]:
async with aiohttp.ClientSession(
    connector=config.tcp_connector_for_concurrency(config.GITHUB_API_CONCURRENCY),
    timeout=aiohttp.ClientTimeout(
        total=config.HTTP_JSON_TIMEOUT_TOTAL,
        connect=config.HTTP_JSON_TIMEOUT_CONNECT,
        sock_connect=config.HTTP_JSON_TIMEOUT_SOCK_CONNECT,
        sock_read=config.HTTP_JSON_TIMEOUT_SOCK_READ,
    ),
) as session:
    await github_graphql.enrich_dataset_with_graphql_info(
        dataset, session, gh_semaphore
    )

In [ ]:
async with aiohttp.ClientSession(
    connector=config.tcp_connector_for_concurrency(config.GITHUB_API_CONCURRENCY),
    timeout=aiohttp.ClientTimeout(
        total=config.HTTP_JSON_TIMEOUT_TOTAL,
        connect=config.HTTP_JSON_TIMEOUT_CONNECT,
        sock_connect=config.HTTP_JSON_TIMEOUT_SOCK_CONNECT,
        sock_read=config.HTTP_JSON_TIMEOUT_SOCK_READ,
    ),
) as session:
    await github_api.enrich_dataset_with_code(dataset, session, gh_semaphore)

checkpoints.save_dataset_checkpoint(dataset, config.DATASET_FINAL_PATH)

In [ ]:
# Compute dataset statistics

num_prs = 0
num_snapshot_commits = 0
num_files_with_comments = 0
num_files_without_comments = 0
num_resolved_comments = 0
num_unresolved_comments = 0
balance_violations = 0  # snapshots where no-comment files exceed commented files
num_files_with_outgoing_deps = 0
total_outgoing_dep_files = 0
num_files_with_incoming_deps = 0
total_incoming_dep_files = 0
num_commits_with_metadata = 0
total_metadata_files = 0
num_commits_with_file_tree = 0
total_file_tree_paths = 0

for repo_name, pr_map in dataset.items():
    for pr_number, pr_entry in pr_map.items():
        num_prs += 1
        for commit_sha, path_map in pr_entry["commits"].items():
            num_snapshot_commits += 1
            snap_with = 0
            snap_without = 0
            for path, file_entry in path_map.items():
                if path == "metadata_files":
                    num_commits_with_metadata += 1
                    total_metadata_files += len(file_entry)
                    continue
                if path == "file_tree":
                    tree_text = file_entry.get("tree", "")
                    if tree_text:
                        num_commits_with_file_tree += 1
                        total_file_tree_paths += len(tree_text.splitlines())
                    continue
                comments = file_entry.get("comments", [])
                for comment in comments:
                    if comment.get("is_resolved", False):
                        num_resolved_comments += 1
                    else:
                        num_unresolved_comments += 1
                if comments:
                    snap_with += 1
                else:
                    snap_without += 1
                out_deps = file_entry.get("outgoing_dependencies", {})
                if out_deps:
                    num_files_with_outgoing_deps += 1
                    total_outgoing_dep_files += len(out_deps)
                in_deps = file_entry.get("incoming_dependencies", {})
                if in_deps:
                    num_files_with_incoming_deps += 1
                    total_incoming_dep_files += len(in_deps)
            num_files_with_comments += snap_with
            num_files_without_comments += snap_without
            if snap_without > snap_with:
                balance_violations += 1

num_files = num_files_with_comments + num_files_without_comments
num_comments = num_resolved_comments + num_unresolved_comments

print(f"Number of PRs:                  {num_prs}")
print(f"Number of snapshot commits:     {num_snapshot_commits}")
print(f"Number of files (total):        {num_files}")
print(f"  - with comments:              {num_files_with_comments}")
print(f"  - without comments:           {num_files_without_comments}")
print(f"Number of comments (total):     {num_comments}")
print(f"  - resolved:                   {num_resolved_comments}")
print(f"  - unresolved:                 {num_unresolved_comments}")

out_dep_pct = 100 * num_files_with_outgoing_deps / num_files if num_files else 0.0
print(
    f"Files with >=1 outgoing dep:    {num_files_with_outgoing_deps} ({out_dep_pct:.1f}%)"
)
print(f"Total outgoing dep entries:     {total_outgoing_dep_files}")

in_dep_pct = 100 * num_files_with_incoming_deps / num_files if num_files else 0.0
print(
    f"Files with >=1 incoming dep:    {num_files_with_incoming_deps} ({in_dep_pct:.1f}%)"
)
print(f"Total incoming dep entries:     {total_incoming_dep_files}")

meta_pct = (
    100 * num_commits_with_metadata / num_snapshot_commits
    if num_snapshot_commits
    else 0.0
)
print(f"Commits with metadata files:    {num_commits_with_metadata} ({meta_pct:.1f}%)")
print(f"Total metadata file entries:    {total_metadata_files}")

file_tree_pct = (
    100 * num_commits_with_file_tree / num_snapshot_commits
    if num_snapshot_commits
    else 0.0
)
print(f"Commits with file tree:         {num_commits_with_file_tree} ({file_tree_pct:.1f}%)")
print(f"Total file-tree path entries:   {total_file_tree_paths}")

In [ ]:
# Convert dataset to JSON with list of files
files_list = []


def _normalize_patched_content(text: str) -> str:
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


raw_rows_total = 0
rows_kept_total = 0
rows_dropped_duplicates = 0
rows_dropped_with_comments = 0
rows_dropped_without_comments = 0

for repo_name, pr_map in dataset.items():
    seen_keys = set()
    for pr_number, pr_entry in pr_map.items():
        pr_title = pr_entry.get("pr_title")
        pr_body = pr_entry.get("pr_body")
        repo_star_count = pr_entry.get("repo_star_count")
        for commit_sha, path_map in pr_entry["commits"].items():
            commit_metadata_files = path_map.get("metadata_files")
            commit_file_tree = (path_map.get("file_tree") or {}).get("tree")
            for path, file_entry in path_map.items():
                if path in {"metadata_files", "file_tree"}:
                    continue
                raw_rows_total += 1
                patched_content = file_entry.get("patched_content", "")
                normalized_patched_content = _normalize_patched_content(patched_content)
                comments = file_entry.get("comments", [])
                has_comments = bool(comments)
                dedupe_key = (path, has_comments, normalized_patched_content)
                if dedupe_key in seen_keys:
                    rows_dropped_duplicates += 1
                    if has_comments:
                        rows_dropped_with_comments += 1
                    else:
                        rows_dropped_without_comments += 1
                    continue
                seen_keys.add(dedupe_key)

                file_obj = {
                    "repo": repo_name,
                    "pr_number": pr_number,
                    "pr_title": pr_title,
                    "pr_body": pr_body,
                    "repo_star_count": repo_star_count,
                    "commit_sha": commit_sha,
                    "path": path,
                    "patched_content": patched_content,
                    "outgoing_dependencies": file_entry.get(
                        "outgoing_dependencies", {}
                    ),
                    "incoming_dependencies": file_entry.get(
                        "incoming_dependencies", {}
                    ),
                    "metadata_files": commit_metadata_files,
                    "file_tree": commit_file_tree,
                    "comments": [
                        {
                            "body": comment.get("body"),
                            "is_resolved": comment.get("is_resolved"),
                            "annotated_start_line": comment.get("annotated_start_line"),
                            "annotated_end_line": comment.get("annotated_end_line"),
                        }
                        for comment in comments
                    ],
                }
                files_list.append(file_obj)
                rows_kept_total += 1

print(f"Export rows before dedupe:       {raw_rows_total}")
print(f"Export rows after dedupe:        {rows_kept_total}")
print(f"Dropped duplicate rows:          {rows_dropped_duplicates}")
print(f"  - dropped with comments:       {rows_dropped_with_comments}")
print(f"  - dropped without comments:    {rows_dropped_without_comments}")

with open("files_list.json", "w") as f:
    json.dump(files_list, f)

In [ ]:
with open("./files_list.json", "r") as f:
    files_list = json.load(f)

In [ ]:
import pandas as pd

df = pd.read_json("./files_list.json")

In [ ]:
df